In [3]:
import os
from getpass import getpass
os.environ["DB_PASSWORD"] = getpass("DB password: ")
import pandas as pd

In [4]:

from db import load_requests
df = load_requests()
df["hour"] = pd.to_datetime(df["time_of_day"]).dt.hour

In [7]:
df["income"].value_counts(normalize=True)

income
0-10k        0.127431
100k-150k    0.125431
250k+        0.125191
40k-60k      0.125161
10k-20k      0.124861
20k-40k      0.124331
60k-100k     0.124251
150k-250k    0.123341
Name: proportion, dtype: float64

In [8]:
df["age"].unique()

<ArrowStringArray>
['46-55', '0-16', '56-65', '26-35', '17-25', '36-45', '76+', '66-75']
Length: 8, dtype: str

In [9]:
for col in ["age", "gender", "is_banned"]:
    print(pd.crosstab(df[col], df["income"], normalize="index").round(2), "\n")

df["hour"] = pd.to_datetime(df["time_of_day"]).dt.hour
print(pd.crosstab(df["hour"], df["income"], normalize="index").round(2))

income  0-10k  100k-150k  10k-20k  150k-250k  20k-40k  250k+  40k-60k  \
age                                                                     
0-16     0.34       0.00     0.33       0.00     0.00   0.33     0.00   
17-25    0.34       0.00     0.33       0.00     0.33   0.00     0.00   
26-35    0.00       0.00     0.33       0.00     0.33   0.00     0.34   
36-45    0.00       0.00     0.00       0.00     0.34   0.00     0.33   
46-55    0.00       0.34     0.00       0.00     0.00   0.00     0.33   
56-65    0.00       0.33     0.00       0.33     0.00   0.00     0.00   
66-75    0.00       0.34     0.00       0.33     0.00   0.34     0.00   
76+      0.34       0.00     0.00       0.33     0.00   0.33     0.00   

income  60k-100k  
age               
0-16        0.00  
17-25       0.00  
26-35       0.00  
36-45       0.33  
46-55       0.33  
56-65       0.34  
66-75       0.00  
76+         0.00   

income  0-10k  100k-150k  10k-20k  150k-250k  20k-40k  250k+  40k-60k  \
gend

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

FEATURES = ["gender", "age", "country", "hour"]
CATEGORICAL = ["gender", "age", "country"]

def model(df:pd.DataFrame, seed=42):
    data = df[["gender", "age", "country", "is_banned", "time_of_day", "income"]].dropna().copy()
    data["hour"] = pd.to_datetime(data["time_of_day"]).dt.hour

    # one-hot code
    X = pd.get_dummies(data[FEATURES], columns=CATEGORICAL)
    y = data["income"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    model = RandomForestClassifier(n_estimators=200, random_state=seed)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)

    results = data.loc[X_test.index, FEATURES].copy()
    results["income"] = y_test
    results["predicted_income"] = pred
    return acc, results



acc, _ = model(df, seed=seed)
print(f"seed={seed:5d}  accuracy={acc:.4f}")

seed=   42  accuracy=0.3411


In [15]:
pd.crosstab([df["age"], df["gender"]], df["income"], normalize="index").round(2)

income        0-10k  100k-150k  10k-20k  150k-250k  20k-40k  250k+  40k-60k  \
age   gender                                                                  
0-16  Female   0.33       0.00     0.33       0.00     0.00   0.34     0.00   
      Male     0.34       0.00     0.33       0.00     0.00   0.32     0.00   
17-25 Female   0.35       0.00     0.33       0.00     0.32   0.00     0.00   
      Male     0.33       0.00     0.33       0.00     0.33   0.00     0.00   
26-35 Female   0.00       0.00     0.33       0.00     0.34   0.00     0.34   
      Male     0.00       0.00     0.33       0.00     0.32   0.00     0.34   
36-45 Female   0.00       0.00     0.00       0.00     0.33   0.00     0.33   
      Male     0.00       0.00     0.00       0.00     0.34   0.00     0.33   
46-55 Female   0.00       0.34     0.00       0.00     0.00   0.00     0.33   
      Male     0.00       0.34     0.00       0.00     0.00   0.00     0.34   
56-65 Female   0.00       0.34     0.00       0.33     0.00   0.00     0.00   
      Male     0.00       0.33     0.00       0.33     0.00   0.00     0.00   
66-75 Female   0.00       0.35     0.00       0.32     0.00   0.33     0.00   
      Male     0.00       0.33     0.00       0.33     0.00   0.34     0.00   
76+   Female   0.35       0.00     0.00       0.32     0.00   0.33     0.00   
      Male     0.33       0.00     0.00       0.34     0.00   0.33     0.00   

income        60k-100k  
age   gender            
0-16  Female      0.00  
      Male        0.00  
17-25 Female      0.00  
      Male        0.00  
26-35 Female      0.00  
      Male        0.00  
36-45 Female      0.34  
      Male        0.33  
46-55 Female      0.33  
      Male        0.33  
56-65 Female      0.33  
      Male        0.34  
66-75 Female      0.00  
      Male        0.00  
76+   Female      0.00  
      Male        0.00

In [17]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
def eval_features(df, features, seeds=range(10)):
    accs = []
    for s in seeds:
        data = df[features + ["income"]].dropna()
        X = pd.get_dummies(data[features], columns=features)
        y = data["income"]
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.2, random_state=s, stratify=y)
        m = DecisionTreeClassifier(random_state=s).fit(X_tr, y_tr)
        accs.append(accuracy_score(y_te, m.predict(X_te)))
    return np.mean(accs), np.std(accs), min(accs), max(accs)

for feats in [["age"], ["age", "gender"], ["age", "country"]]:
    mean, std, lo, hi = eval_features(df, feats)
    print(f"{str(feats):25s} mean={mean:.4f} std={std:.4f} range=[{lo:.4f}, {hi:.4f}]")

['age']                   mean=0.3362 std=0.0019 range=[0.3333, 0.3387]
['age', 'gender']         mean=0.3362 std=0.0037 range=[0.3316, 0.3437]
['age', 'country']        mean=0.3340 std=0.0025 range=[0.3270, 0.3361]
